# 08 — Telephony — Plivo

V3 webhook signature (HMAC-SHA256), Plivo XML answer, AMD, native DTMF send (the parity gain over Twilio), ring timeout, status callback, voicemail drop. Plivo streams mulaw 8 kHz over the same WebSocket-media model as Twilio — we pin the codec in the answer XML.


## Optional: run in Docker

Uncomment the cell below to launch JupyterLab + EmbeddedServer in a container. It's idempotent and a no-op once you're already inside the container. See `examples/notebooks/python/Dockerfile` and `docker-compose.yml` for what it builds.


In [ ]:
# Optional — launch the patter-notebooks Docker stack from this cell.
# Skip this cell to run on your host Python. Idempotent if uncommented.
#
# import _setup
# _setup.start_docker()             # build + up -d, prints http://localhost:8888
# _setup.start_docker(open_url=True)  # …and open the browser tab


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | _none — offline crypto demos_ |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + Plivo creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Plivo carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Plivo
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Plivo(
                auth_id='MA-test-only',
                auth_token='test',  # gitleaks:allow
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        assert p._local_config.telephony_provider == 'plivo'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Plivo, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Plivo(auth_id='MA-test-only', auth_token='test'),  # gitleaks:allow
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


## §2: Feature Tour

Exercises Plivo carrier construction, V3 webhook signature verification (HMAC-SHA256 of `url + sorted_post_params + "." + nonce`), the Plivo answer XML, and the AMD classifier. All offline.


### V3 signature — POST with form params (the real production case)

Plivo's V3 scheme:

* **POST**: `signed = url + sorted_post_params + "." + nonce`
  Params are sorted alphabetically by key (case-sensitive) and concatenated as `key1value1key2value2…` with no delimiters.
* **GET**:  `signed = url + "." + nonce`

HMAC-SHA256 keyed on the account `auth_token`, base64-encoded. The `X-Plivo-Signature-V3` header may carry multiple comma-separated signatures during key rotation.


In [ ]:
import hmac, hashlib, base64
from getpatter.server import _validate_plivo_signature
with _setup.cell('plivo_v3_sig_post', tier=1, env=env) as ok:
    if ok:
        auth_token = 'test_plivo_auth_token_32chars___'  # gitleaks:allow
        url   = 'https://example.com/webhooks/plivo/voice'
        nonce = 'b3a1f0c4-2025-4f6e-9a52-7e34a8b8e0c2'
        params = {'CallUUID': 'CU_xyz', 'From': '+15551112222', 'To': '+15553334444'}
        sorted_concat = ''.join(f'{k}{params[k]}' for k in sorted(params))
        signed = f'{url}{sorted_concat}.{nonce}'
        sig = base64.b64encode(
            hmac.new(auth_token.encode(), signed.encode(), hashlib.sha256).digest()
        ).decode()
        valid = _validate_plivo_signature(url, nonce, sig, auth_token, params=params, method='POST')
        print(f'signed prefix : {signed[:80]}...')
        print(f'signature     : {sig}')
        print(f'valid         : {valid}')
        assert valid, 'POST signature should be valid'


### V3 signature — GET (e.g. the `/webhooks/plivo/transfer` aleg_url)

Query params live in the URL already, so the signed string is just `url + "." + nonce` — no separate param concatenation.


In [ ]:
import hmac, hashlib, base64
from getpatter.server import _validate_plivo_signature
with _setup.cell('plivo_v3_sig_get', tier=1, env=env) as ok:
    if ok:
        auth_token = 'test_plivo_auth_token_32chars___'  # gitleaks:allow
        url   = 'https://example.com/webhooks/plivo/transfer?to=%2B15551234567'
        nonce = 'b3a1f0c4-2025-4f6e-9a52-7e34a8b8e0c2'
        sig = base64.b64encode(
            hmac.new(auth_token.encode(), f'{url}.{nonce}'.encode(), hashlib.sha256).digest()
        ).decode()
        valid = _validate_plivo_signature(url, nonce, sig, auth_token, method='GET')
        print(f'valid: {valid}')
        assert valid


### V3 signature — tampered request rejected

Tampering with any POST param value invalidates the signature.


In [ ]:
import hmac, hashlib, base64
from getpatter.server import _validate_plivo_signature
with _setup.cell('plivo_v3_sig_tampered', tier=1, env=env) as ok:
    if ok:
        auth_token = 'test_plivo_auth_token_32chars___'  # gitleaks:allow
        url, nonce = 'https://example.com/webhooks/plivo/voice', 'n'
        original = {'CallUUID': 'CU1', 'From': '+15551112222'}
        sorted_concat = ''.join(f'{k}{original[k]}' for k in sorted(original))
        sig = base64.b64encode(
            hmac.new(auth_token.encode(), f'{url}{sorted_concat}.{nonce}'.encode(), hashlib.sha256).digest()
        ).decode()
        tampered = {'CallUUID': 'CU1', 'From': '+15559999999'}
        valid = _validate_plivo_signature(url, nonce, sig, auth_token, params=tampered, method='POST')
        print(f'tampered signature valid: {valid}  (expected False)')
        assert not valid, 'tampered signature must be rejected'


### Plivo answer XML

Unlike Twilio (`<Stream url=…>`), Plivo embeds the WSS URL as the `<Stream>` element's **text content**. The `&` between query parameters MUST be escaped to `&amp;` or Plivo truncates the URL at the first parameter.


In [ ]:
from getpatter.providers.plivo_adapter import PlivoAdapter
with _setup.cell('plivo_answer_xml', tier=1, env=env) as ok:
    if ok:
        xml = PlivoAdapter.generate_stream_xml(
            'wss://example.com/ws/plivo/stream/CALLUUID?caller=%2B1&callee=%2B2',
            content_type='audio/x-mulaw;rate=8000',
            extra_headers={'X-PH-caller': '+1', 'X-PH-callee': '+2'},
        )
        print(xml)
        assert '<Stream' in xml and 'bidirectional="true"' in xml
        assert '&amp;callee' in xml, '& must be XML-escaped'
        assert 'audio/x-mulaw;rate=8000' in xml


### AMD classifier — defensive across Plivo result spellings

Plivo's async machine-detection callback reports the outcome via a result field whose spelling varies by API version. `_classify_plivo_amd` normalises common shapes to `human` / `machine` / `fax` / `unknown`.


In [ ]:
from getpatter.server import _classify_plivo_amd
with _setup.cell('plivo_amd_classify', tier=1, env=env) as ok:
    if ok:
        cases = [
            ('human',              'human'),
            ('person',             'human'),
            ('machine',            'machine'),
            ('machine_end_beep',   'machine'),
            ('answering_machine',  'machine'),
            ('true',               'machine'),
            ('fax',                'fax'),
            ('',                   'unknown'),
            ('weird',              'unknown'),
        ]
        for raw, expected in cases:
            got = _classify_plivo_amd(raw)
            status = '✓' if got == expected else '✗'
            print(f'  {status} {raw!r:22s} → {got}')
        assert all(_classify_plivo_amd(r) == e for r, e in cases)


### Native DTMF send — a parity *gain* over Twilio

Plivo accepts a `sendDTMF` command over the media WebSocket — Twilio Media Streams cannot send DTMF at all. `PlivoAudioSender.send_dtmf` filters invalid characters and emits the JSON frame.


In [ ]:
import json
from getpatter.telephony.plivo import PlivoAudioSender
with _setup.cell('plivo_send_dtmf', tier=1, env=env) as ok:
    if ok:
        class FakeWs:
            def __init__(self): self.sent = []
            async def send_text(self, t): self.sent.append(json.loads(t))
        import asyncio
        ws = FakeWs()
        sender = PlivoAudioSender(ws, 'stream-xyz', input_is_mulaw_8k=True)
        asyncio.get_event_loop().run_until_complete(sender.send_dtmf('12ab#xZ'))
        # Z is not in the allowed set 0-9 * # A-D a-d w W; everything else passes.
        print(ws.sent[-1])
        assert ws.sent[-1] == {'event': 'sendDTMF', 'dtmf': '12ab#'}


### E.164 phone number patterns


In [ ]:
import re
with _setup.cell('e164_patterns', tier=1, env=env) as ok:
    if ok:
        E164_RE = re.compile(r'^\+[1-9]\d{6,14}$')
        cases = [
            ('+15555550100', True),
            ('+442071838750', True),
            ('+393399123456', True),
            ('15555550100',  False),
            ('+1',          False),
            ('not-a-number', False),
        ]
        for number, expected in cases:
            result = bool(E164_RE.match(number))
            status = '✓' if result == expected else '✗'
            print(f'  {status} {number!r:20s} → {result}')
        assert all(bool(E164_RE.match(n)) == e for n, e in cases)


### Plivo carrier construction


In [ ]:
from getpatter import Patter, Plivo
with _setup.cell('plivo_carrier', tier=1, env=env) as ok:
    if ok:
        carrier = Plivo(
            auth_id='MA-test-only',
            auth_token='test_token',  # gitleaks:allow
        )
        p = Patter(
            carrier=carrier,
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        lc = p._local_config
        print(f'carrier:  {lc.telephony_provider}')
        print(f'phone:    {lc.phone_number}')
        print(f'webhook:  {lc.webhook_url}')
        assert lc.telephony_provider == 'plivo'


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.

Tests Plivo's outbound call flow including AMD detection. Requires `ENABLE_LIVE_CALLS=1` plus `PLIVO_AUTH_ID`, `PLIVO_AUTH_TOKEN`, `PLIVO_PHONE_NUMBER`, `TARGET_PHONE_NUMBER` and a public webhook URL.


### Pre-flight checklist


In [ ]:
with _setup.cell(
    'live_preflight',
    tier=4,
    required=['PLIVO_AUTH_ID', 'PLIVO_AUTH_TOKEN', 'PLIVO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER'],
    env=env,
) as ok:
    if ok:
        print(f'  carrier:  Plivo {env.plivo_number}  →  {env.target_number}')
        print(f'  webhook:  {env.public_webhook_url or "(ngrok auto-launch)"}')
        print(f'  features: V3 signature + AMD + voicemail fallback')


### Live Plivo call with AMD *(T4)*


In [ ]:
from getpatter import Patter, Plivo, OpenAIRealtime
with _setup.cell(
    'live_plivo_amd',
    tier=4,
    required=['PLIVO_AUTH_ID', 'PLIVO_AUTH_TOKEN', 'PLIVO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'],
    env=env,
) as ok:
    if ok:
        p = Patter(
            carrier=Plivo(auth_id=env.plivo_auth_id, auth_token=env.plivo_auth_token),
            phone_number=env.plivo_number,
            webhook_url=env.public_webhook_url,
        )
        agent = p.agent(
            system_prompt='You are a Plivo telephony demo agent. Greet the caller and hang up.',
            engine=OpenAIRealtime(api_key=env.openai_key),
        )
        await p.call(
            env.target_number,
            agent=agent,
            machine_detection=True,
            voicemail_message='You reached Patter demo. Goodbye.',
            first_message='Hello from Patter Plivo demo.',
            ring_timeout=env.max_call_seconds,
        )
        print('✓ Plivo AMD call completed')
